# Importacion de librerias y conexión

In [119]:
import mysql.connector
import pandas as pd
import numpy as np
import regex as re
from datetime import datetime

In [62]:
conn = mysql.connector.connect(
    host='212.227.90.6',
    user='Equipo19',
    password='E1q2u3i4p5o19',
    database='Equip_19'
)

cursor = conn.cursor()

cursor.execute("SHOW TABLES")
tablas = [t[0] for t in cursor.fetchall()]


dfs = {tabla: pd.read_sql(f"SELECT * FROM {tabla}", conn) for tabla in tablas}

df = dfs['Tourist_Accommodation03112025']

C:\Users\irene\AppData\Local\Temp\ipykernel_33496\3397124836.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dfs = {tabla: pd.read_sql(f"SELECT * FROM {tabla}", conn) for tabla in tablas}


In [63]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 35 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   apartment_id                 8000 non-null   int64  
 1   name                         7997 non-null   object 
 2   description                  7946 non-null   object 
 3   host_id                      8000 non-null   int64  
 4   neighbourhood_name           8000 non-null   object 
 5   neighbourhood_district       4861 non-null   object 
 6   room_type                    8000 non-null   object 
 7   accommodates                 8000 non-null   int64  
 8   bathrooms                    7957 non-null   object 
 9   bedrooms                     7961 non-null   object 
 10  beds                         7992 non-null   float64
 11  amenities_list               7983 non-null   object 
 12  price                        7829 non-null   float64
 13  minimum_nights    

# Tratamiento de nulos

In [64]:
# 1. Definir la condición de "Precio Inválido" (Nulo O Cero)
precio_invalido = df['price'].isnull() | (df['price'] == 0)

# 2. Filtrar el DataFrame original para obtener solo los registros con precio inválido
df_precios_invalidos = df.loc[precio_invalido].copy()

# 3. Analizar los Precios Inválidos (como lo hacías)
print(f"✅ Total de registros con precio inválido (Nulos O Cero): {len(df_precios_invalidos)}")
print(f"   Filas con precio nulo (NaN): {df_precios_invalidos['price'].isnull().sum()}")
print(f"   Filas con precio a cero (0): {(df_precios_invalidos['price'] == 0).sum()}")

# 4. **NUEVA VERIFICACIÓN:** Encontrar los ID de apartamentos duplicados dentro de este subconjunto
duplicados_en_invalidos = df_precios_invalidos.duplicated(subset=['apartment_id'], keep=False)

# 5. Filtrar el subconjunto para mostrar SÓLO los alojamientos que tienen:
#    a) Precio inválido (ya filtrado)
#    b) Y un 'apartment_id' duplicado
df_apartamentos_duplicados_y_invalidos = df_precios_invalidos.loc[duplicados_en_invalidos]

print("\n--- Resultados de Duplicados ---")
print(f"⚠️ Total de alojamientos con precio inválido Y ID duplicado: {len(df_apartamentos_duplicados_y_invalidos)}")


✅ Total de registros con precio inválido (Nulos O Cero): 171
   Filas con precio nulo (NaN): 171
   Filas con precio a cero (0): 0

--- Resultados de Duplicados ---
⚠️ Total de alojamientos con precio inválido Y ID duplicado: 20


# Conversión de Tipos

CONVERSIÓN DE FECHAS

In [65]:
df["first_review_date"] = pd.to_datetime(df["first_review_date"], errors="coerce")
df["last_review_date"] = pd.to_datetime(df["last_review_date"], errors="coerce")
df["insert_date"] = pd.to_datetime(df["insert_date"], errors="coerce")

C:\Users\irene\AppData\Local\Temp\ipykernel_33496\3908789225.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["insert_date"] = pd.to_datetime(df["insert_date"], errors="coerce")


CONVERSIÓN A BOOLEANO

In [66]:
df['has_availability'] = df['has_availability'].replace({'VERDADERO': True})
df['has_availability'] = df['has_availability'].astype(bool)

df['is_instant_bookable'] = df['is_instant_bookable'].replace({'VERDADERO': True, 'FALSO': False})
df['is_instant_bookable'] = df['is_instant_bookable'].astype(bool)

C:\Users\irene\AppData\Local\Temp\ipykernel_33496\1651559890.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_instant_bookable'] = df['is_instant_bookable'].replace({'VERDADERO': True, 'FALSO': False})


CONVERSIÓN A INTEGRO

In [67]:
df['bathrooms'] = pd.to_numeric(df['bathrooms'], errors='coerce').astype('Int64')
df["bedrooms"] = pd.to_numeric(df["bedrooms"], errors='coerce').astype('Int64')
df['beds'] = pd.to_numeric(df['beds'], errors='coerce').astype('Int64')
df['is_instant_bookable'] = df['is_instant_bookable'].astype(int)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 35 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   apartment_id                 8000 non-null   int64         
 1   name                         7997 non-null   object        
 2   description                  7946 non-null   object        
 3   host_id                      8000 non-null   int64         
 4   neighbourhood_name           8000 non-null   object        
 5   neighbourhood_district       4861 non-null   object        
 6   room_type                    8000 non-null   object        
 7   accommodates                 8000 non-null   int64         
 8   bathrooms                    7957 non-null   Int64         
 9   bedrooms                     7961 non-null   Int64         
 10  beds                         7992 non-null   Int64         
 11  amenities_list               7983 non-null 

# Duplicados

COMPROBACIÓN DUPLICADOS APARTMENT_ID

In [68]:
df['apartment_id'].duplicated().any()

np.True_

In [69]:
total_duplicados = df['apartment_id'].duplicated().sum()
print(f"Total de registros duplicados encontrados en 'apartment_id': {total_duplicados}")

Total de registros duplicados encontrados en 'apartment_id': 307


ELIMINACIÓN DE DUPLICADOS DE APARTMENT_ID, MANTENIENDO LOS MÁS RECIENTES SEGÚN INSERT_DATE

In [70]:
df[df['apartment_id'].duplicated(keep=False)]

df = df.sort_values(['apartment_id', 'insert_date'])

df = df.drop_duplicates(subset='apartment_id', keep='last')

In [71]:
df['apartment_id'].duplicated().any()

np.False_

In [72]:
df.shape

(7693, 35)

# Estandarización de categorías y texto

NEIGHBOURHOOD

In [73]:
def clean_neighbourhood(text):
    if pd.isna(text) or text == '':
        return pd.NA
    text = str(text)
    text = text.replace("�", "")
    text = re.sub(r"[^\p{L}\p{N}\s'\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = text.title()
    
    return text if text else pd.NA

In [74]:
df['neighbourhood_name'] = df['neighbourhood_name'].apply(clean_neighbourhood)
df['neighbourhood_district'] = df['neighbourhood_district'].apply(clean_neighbourhood)

In [75]:
df[['neighbourhood_name', 'neighbourhood_district']].head(10)

,neighbourhood_name,neighbourhood_district
0,Centro,<NA>
1,Crmenes,Latina
2,San Vicente,Casco Antiguo
3,L'Antiga Esquerra De L'Eixample,Eixample
4,Quart,<NA>
5,Torroella De Fluvi,<NA>
6,El Camp De L'Arpa Del Clot,Sant Mart
7,La Dreta De L'Eixample,Eixample
8,Embajadores,Centro
9,El Camp D'En Grassot I Grcia Nova,Grcia


ESTANDARIZAR CITY

In [76]:
df['city'].value_counts()

city
barcelona    2261
madrid       1645
mallorca     1265
girona       1198
sevilla       403
malaga        394
valencia      360
menorca       167
Name: count, dtype: int64

In [77]:
city_mapping = {
    'Seville': 'Sevilla',
    'Madrid': 'Madrid',
    'Barcelona': 'Barcelona',
    'Valencia': 'Valencia',
    'Malaga': 'Málaga',
    'Málaga': 'Málaga',        
    'València': 'Valencia',    
    'Sevilla': 'Sevilla',
    'barcelona': 'Barcelona',
    'MADRID': 'Madrid',
    'valencia': 'Valencia',
    'malaga': 'Málaga',
    'sevilla': 'Sevilla',
    'girona': 'Girona',
    'Girona': 'Girona',
    'mallorca': 'Mallorca',
    'Mallorca': 'Mallorca',
    'Menorca': 'Menorca',
    'menorca': 'Menorca',
    'madrid': 'Madrid',
    'barcelona': 'Barcelona',
    'valencia': 'Valencia',
    'malaga': 'Málaga',
    'sevilla': 'Sevilla',
    'girona': 'Girona',
    'mallorca': 'Mallorca',
    'menorca': 'Menorca',
}

In [78]:
valid_cities = {
    'Barcelona',
    'Madrid',
    'Sevilla',
    'Valencia',
    'Málaga',
    'Girona',
    'Mallorca',
    'Menorca'
}

In [79]:
df['city_clean'] = df['city'].astype(str).str.strip()

# Crea una versión en minúsculas del mapeo para la coincidencia que no distinga entre mayúsculas y minúsculas.
mapping_lower = {k.lower(): v for k, v in city_mapping.items()}

# Normalizar
df['city_normalized'] = df['city_clean'].str.lower().map(mapping_lower)

# Filtrar solo cidades válidas
df = df[df['city_normalized'].isin(valid_cities)]

# Substituir colmuna original
df['city'] = df['city_normalized']
df = df.drop(columns=['city_clean', 'city_normalized'])

In [80]:
df[['city']].head(5)

,city
0,Málaga
1,Madrid
2,Sevilla
3,Barcelona
4,Girona


ESTANDARIZAR COUNTRY

In [81]:
df['country'] = 'Spain'
df[['country']].head(5)

,country
0,Spain
1,Spain
2,Spain
3,Spain
4,Spain


# Tratamiento de valores nulos

In [82]:
df.shape

(7693, 35)

PRECIO

In [83]:
precio_invalido = df['price'].isnull() | (df['price'] == 0)

df_precios_invalidos = df.loc[precio_invalido]

print(f"Total de registros inválidos (Nulos O Cero): {len(df_precios_invalidos)}")
print(f"Filas con precio nulo (NaN): {df['price'].isnull().sum()}")
print(f"Filas con precio a cero (0): {(df['price'] == 0).sum()}")

Total de registros inválidos (Nulos O Cero): 159
Filas con precio nulo (NaN): 159
Filas con precio a cero (0): 0


PRICE, CITY Y ROOM_TYPE

In [84]:
# Marcar inválidos
df['is_valid'] = True
df.loc[df['price'].isna() | df['city'].isna() | df['room_type'].isna(), 'is_valid'] = False
df_valid = df[df['is_valid']].copy()

In [85]:
df = df_valid

In [86]:
df.shape

(7534, 36)

Nulos en variables reviews


In [87]:
# Forzar nulos en reviews si no hay reviews
review_cols = [
    'first_review_date', 'last_review_date',
    'review_scores_rating', 'review_scores_accuracy',
    'review_scores_cleanliness', 'review_scores_checkin',
    'review_scores_communication', 'review_scores_location',
    'review_scores_value', 'reviews_per_month'
]
mask_no_reviews = df_valid['number_of_reviews'] == 0
df_valid.loc[mask_no_reviews, review_cols] = np.nan

# Imputar reviews_per_month
def calculate_reviews_per_month(row):
    if row['number_of_reviews'] > 0:
        if not pd.isna(row['first_review_date']):
            today = datetime.now()
            days_since_first = (today - row['first_review_date']).days
            if days_since_first > 0:
                return row['number_of_reviews'] / (days_since_first / 30)
        elif not pd.isna(row['last_review_date']):
            today = datetime.now()
            days_since_last = (today - row['last_review_date']).days
            if days_since_last > 0:
                return row['number_of_reviews'] / (days_since_last / 30)
    return np.nan

df_valid['reviews_per_month'] = df_valid.apply(calculate_reviews_per_month, axis=1)

# Amenities_list
df_valid['amenities_list'] = df_valid['amenities_list'].apply(lambda x: [] if pd.isna(x) or (isinstance(x, str) and x.strip() == '') else x)

# Validar coherencia
review_cols_no_nulls = df_valid.loc[(df_valid['number_of_reviews'] == 0) & (df_valid[review_cols].notna().any(axis=1))]
print(f"Registros inconsistentes: {len(review_cols_no_nulls)}")

Registros inconsistentes: 0


# Coherencia entre variables

COMPROBAR QUE MIN NIGHTS < MAX NIGHTS

In [88]:
df_error = df[df['minimum_nights'] > df['maximum_nights']]
df_error
# Sin resultados, así que todo está correcto

,apartment_id,name,description,host_id,neighbourhood_name,neighbourhood_district,room_type,accommodates,bathrooms,bedrooms,beds,amenities_list,price,minimum_nights,maximum_nights,has_availability,availability_30,availability_60,availability_90,availability_365,number_of_reviews,first_review_date,last_review_date,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,is_instant_bookable,reviews_per_month,country,city,insert_date,is_valid


COMPROBAR AVAILABILITY

In [89]:
df_hav = df[
    (df['availability_30'] > df['availability_60']) &
    (df['availability_60'] > df['availability_90']) &
    (df['availability_90'] > df['availability_365'])
].copy()

df_hav

# Sin resultados, así que todo está correcto

,apartment_id,name,description,host_id,neighbourhood_name,neighbourhood_district,room_type,accommodates,bathrooms,bedrooms,beds,amenities_list,price,minimum_nights,maximum_nights,has_availability,availability_30,availability_60,availability_90,availability_365,number_of_reviews,first_review_date,last_review_date,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,is_instant_bookable,reviews_per_month,country,city,insert_date,is_valid


In [90]:
cols_disp = ['availability_30', 'availability_60', 'availability_90', 'availability_365']

condicion_invalida = (
    (df['has_availability'] == True) & 
    (df[cols_disp].isna().all(axis=1))
)

df_invalidas = df[condicion_invalida]
print(df_invalidas)

# Empty Dataframe, así que todo está correcto

Empty DataFrame
Columns: [apartment_id, name, description, host_id, neighbourhood_name, neighbourhood_district, room_type, accommodates, bathrooms, bedrooms, beds, amenities_list, price, minimum_nights, maximum_nights, has_availability, availability_30, availability_60, availability_90, availability_365, number_of_reviews, first_review_date, last_review_date, review_scores_rating, review_scores_accuracy, review_scores_cleanliness, review_scores_checkin, review_scores_communication, review_scores_location, review_scores_value, is_instant_bookable, reviews_per_month, country, city, insert_date, is_valid]
Index: []


LIMPIEZA AVAILABILITY

In [91]:
# Comprobamos coherencia entre los datos de las columnas has_availability y las cols_disp:
df_disponibilidad = df.loc[df['has_availability'] != True, 
                     ['has_availability', 'availability_30', 'availability_60', 'availability_90', 'availability_365']]

df_disponibilidad
# No hay coherencia en estos casos

,has_availability,availability_30,availability_60,availability_90,availability_365
7,False,28,55,85,360
12,False,7,23,53,328
29,False,13,43,72,347
32,False,5,11,37,234
36,False,8,14,25,224
...,...,...,...,...,...
4762,False,16,46,76,76
4771,False,6,15,31,114
4791,False,13,43,73,163
4802,False,3,33,63,338


In [92]:
# Se define la condición de disponibilidad:
sidisponible = (df[cols_disp] > 0).any(axis=1)

# Se corrige la columna 'has_availability' en función de las cols_disp:
df.loc[(df['has_availability'] == False) & sidisponible, 'has_availability'] = True

In [93]:
# Comprobamos:
df[['has_availability'] + cols_disp].head()

,has_availability,availability_30,availability_60,availability_90,availability_365
0,True,7,20,40,130
1,True,0,0,0,162
2,True,26,31,31,270
3,True,9,23,49,300
4,True,0,19,49,312


COHERENCIA FECHAS INSERT_DATE ANTES QUE FIRST_REVIEW_DATE

In [94]:
df_fechas = df[
    (df['insert_date'] > df['first_review_date']) 
].copy()

totaldf = len(df_fechas)
print("Hay", totaldf, "casos en los que la 'insert_date' es posterior a la 'first_review_date', \n por lo que deducimos que se trata de anuncios que han sido actualizados y conservan reseñas antiguas.")


Hay 2435 casos en los que la 'insert_date' es posterior a la 'first_review_date', 
 por lo que deducimos que se trata de anuncios que han sido actualizados y conservan reseñas antiguas.


COHERENCIA FECHAS FIRST_REVIEW DATE ANTES QUE LAST_REVIEW_DATE

In [95]:
df_fechas2 = df[
    (df['first_review_date'] > df['last_review_date'])
].copy()

totaldf2 = len(df_fechas2)

print("Hay", totaldf2, "casos en los que la 'first_review_date' es posterior " \
"a la 'last_review_date', \n por lo que puede que haya habido un error en la adquisición de los datos") 

Hay 71 casos en los que la 'first_review_date' es posterior a la 'last_review_date', 
 por lo que puede que haya habido un error en la adquisición de los datos


In [96]:
# Se intercambian los datos entre las columnas first_review_date' y 'last_review_date' en los casos de incoherencia para deshacer esta inconsistencia  
def arreglar_fechas_reviews(row):
    if row['first_review_date'] > row['last_review_date']:
        row['first_review_date'], row['last_review_date'] = row['last_review_date'], row['first_review_date']
    return row

df = df.apply(arreglar_fechas_reviews, axis=1)

# Cambio de la presentación del dato

CAMBIO DE SCORES RATING A 0 - 100 Y TODOS LOS DEMÁS A 0 - 10

In [97]:
df['review_scores_rating'] = df['review_scores_rating'] / 10
df['review_scores_accuracy'] = df['review_scores_accuracy'] / 10
df['review_scores_checkin'] = df['review_scores_checkin'] / 10
df['review_scores_cleanliness'] = df['review_scores_cleanliness'] / 10
df['review_scores_communication'] = df['review_scores_communication'] / 10
df['review_scores_location'] = df['review_scores_location'] / 10
df['review_scores_value'] = df['review_scores_value'] / 10


In [98]:
df['review_scores_value'].head(10)

0    10.0
1     9.0
2    10.0
3     9.0
4    10.0
5    10.0
6     9.0
7     9.0
8     9.0
9     9.0
Name: review_scores_value, dtype: float64

# Nulos

ELIMINACIÓN NULOS EN BEDROOMS

In [99]:
bedrooms_nulas = df.loc[df['bedrooms'].isnull(), ['bedrooms', 'room_type', 'accommodates']]
print("Hay", len(bedrooms_nulas), "casos nulos en 'bedrooms'.")


Hay 39 casos nulos en 'bedrooms'.


In [100]:
# 1. Imputación para 'Private room' (Hay una sola habitación)
df.loc[(df['room_type'] == 'Private room') & (df['bedrooms'].isnull()), 'bedrooms'] = 1

# 2. Imputación para 'Entire home/apt' (Se calcula en base a los acommodates, para poner una habitación para acda 2, redondeo hacia arriba)
mask_entire = (df['room_type'] == 'Entire home/apt') & (df['bedrooms'].isnull())

# Se imputa el número de habitaciones como la capacidad dividida por 2, redondeado hacia arriba
df.loc[mask_entire, 'bedrooms'] = (df.loc[mask_entire, 'accommodates'] / 2).apply(np.ceil)

In [101]:
# Comprobación
df[['bedrooms', 'room_type', 'accommodates']]

,bedrooms,room_type,accommodates
0,1,Private room,2
1,1,Private room,1
2,2,Entire home/apt,4
3,1,Private room,2
4,2,Private room,5
...,...,...,...
7995,1,Private room,1
7996,3,Entire home/apt,6
7997,1,Entire home/apt,2
7998,2,Private room,3


ELIMINACIÓN NULOS EN BATHROOMS

In [102]:
df.loc[df['bathrooms'].isnull(), ['bathrooms', 'room_type', 'accommodates']]
df.loc[df['bathrooms'].isnull(), 'bathrooms'] = 1

ELIMINACIÓN NULOS EN BEDS

In [103]:
# Damos el valor a beds en función de 'acommodates':

# 1. Crear la máscara (condición) para todos los nulos en la columna 'beds'
mask_beds_null = df['beds'].isnull()

# 2. Aplicar la imputación: Capacidad de personas / 2, redondeando hacia arriba.
df.loc[mask_beds_null, 'beds'] = (df.loc[mask_beds_null, 'accommodates'] / 2).apply(np.ceil)


In [104]:
# Comprobamos:
df[['bedrooms', 'beds', 'accommodates']]

,bedrooms,beds,accommodates
0,1,1,2
1,1,1,1
2,2,2,4
3,1,1,2
4,2,5,5
...,...,...,...
7995,1,1,1
7996,3,4,6
7997,1,2,2
7998,2,2,3


ELIMINACIÓN DE NULOS EN REVIEWS

In [105]:
df.loc[df['reviews_per_month'].isnull(), ['reviews_per_month', 'number_of_reviews']]

,reviews_per_month,number_of_reviews
8,NaN,108
26,NaN,97
27,NaN,44
28,NaN,189
30,NaN,33
...,...,...
7992,NaN,0
7993,NaN,0
7995,NaN,0
7997,NaN,0


In [106]:
# se les da valor 0
df.loc[
    (df['reviews_per_month'].isnull()) & (df['number_of_reviews'] == 0),
    'reviews_per_month'
] = 0

In [107]:
# Comprobamos:
df.loc[df['number_of_reviews'] == 0, 
       ['reviews_per_month', 
        'number_of_reviews', 
        'first_review_date', 
        'last_review_date', 
        'review_scores_rating', 
        'review_scores_accuracy', 
        'review_scores_checkin', 
        'review_scores_cleanliness', 
        'review_scores_communication', 
        'review_scores_location', 
        'review_scores_value']]

,reviews_per_month,number_of_reviews,first_review_date,last_review_date,review_scores_rating,review_scores_accuracy,review_scores_checkin,review_scores_cleanliness,review_scores_communication,review_scores_location,review_scores_value
61,0.0,0,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
103,0.0,0,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
137,0.0,0,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
142,0.0,0,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
168,0.0,0,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
7991,0.0,0,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7992,0.0,0,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7993,0.0,0,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7995,0.0,0,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [108]:
colsrev = [
    'reviews_per_month',
    'number_of_reviews',
    'review_scores_rating',
    'review_scores_accuracy',
    'review_scores_checkin',
    'review_scores_cleanliness',
    'review_scores_communication',
    'review_scores_location',
    'review_scores_value'
]

df.loc[df['number_of_reviews'] == 0, colsrev] = 0

In [109]:
df.isnull().sum()

apartment_id                      0
name                              3
description                      49
host_id                           0
neighbourhood_name                0
neighbourhood_district         2927
room_type                         0
accommodates                      0
bathrooms                         0
bedrooms                          0
beds                              0
amenities_list                    0
price                             0
minimum_nights                    0
maximum_nights                    0
has_availability                  0
availability_30                   0
availability_60                   0
availability_90                   0
availability_365                  0
number_of_reviews                 0
first_review_date              5001
last_review_date               5028
review_scores_rating             83
review_scores_accuracy           92
review_scores_cleanliness        86
review_scores_checkin            97
review_scores_communication 

¿FIRST_REVIEW_DATE Y LAST_REVIEW_DATE?

In [110]:
# df.loc[
#     df['last_review_date'].isnull() & df['first_review_date'].notnull(),
#     ['first_review_date', 'last_review_date', 'number_of_reviews', 'reviews_per_month']
# ]

df.loc[
    df['first_review_date'].isnull() & df['last_review_date'].notnull(),
    ['first_review_date', 'last_review_date', 'number_of_reviews', 'reviews_per_month']
]

,first_review_date,last_review_date,number_of_reviews,reviews_per_month
3,NaT,2020-04-01,292,4.285714
7,NaT,2017-05-06,33,0.318841
11,NaT,2016-01-10,189,1.580708
19,NaT,2017-09-12,154,1.552419
32,NaT,2017-02-04,416,3.904881
...,...,...,...,...
7976,NaT,2020-02-03,35,0.499524
7981,NaT,2020-02-01,13,0.185361
7985,NaT,2020-10-03,13,0.209790
7987,NaT,2019-05-05,10,0.126263


COMPROBACIÓN NULOS DE NEIGHBOURHOOD DISTRICT

In [111]:
df.loc[df["neighbourhood_district"].isna(), "neighbourhood_name"].value_counts()

df_nulls = df[df["neighbourhood_district"].isna()][["neighbourhood_name", "city", "apartment_id"]]

df_nulls

,neighbourhood_name,city,apartment_id
0,Centro,Málaga,11964
4,Quart,Girona,35801
5,Torroella De Fluvi,Girona,48764
27,Palma De Mallorca,Mallorca,164621
30,Alcdia,Mallorca,178456
...,...,...,...
7989,Castell D'Empries,Girona,32361345
7992,Llucmajor,Mallorca,32379297
7994,Santany,Mallorca,32391703
7997,Felanitx,Mallorca,32395123


In [112]:
df[["neighbourhood_district", "neighbourhood_name"]]

,neighbourhood_district,neighbourhood_name
0,<NA>,Centro
1,Latina,Crmenes
2,Casco Antiguo,San Vicente
3,Eixample,L'Antiga Esquerra De L'Eixample
4,<NA>,Quart
...,...,...
7995,Eixample,Sant Antoni
7996,Casco Antiguo,Arenal
7997,<NA>,Felanitx
7998,Sant Mart,Provenals Del Poblenou


# Nuevas columnas

CREACIÓN DE NUEVA COLUMNA

In [113]:
df['ocupacion_mes'] = 30 - df['availability_30']

# EDA

COMPROBACIÓN OUTLIERS

In [114]:
pd.set_option('display.max_columns', None)
df[df['price'] > 6000]

,apartment_id,name,description,host_id,neighbourhood_name,neighbourhood_district,room_type,accommodates,bathrooms,bedrooms,beds,amenities_list,price,minimum_nights,maximum_nights,has_availability,availability_30,availability_60,availability_90,availability_365,number_of_reviews,first_review_date,last_review_date,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,is_instant_bookable,reviews_per_month,country,city,insert_date,is_valid,ocupacion_mes
2390,12970898,APARTAMENTO CENTRICO-EL CARMEN,Apartamento totalmente reformado en 2018. Dos ...,71363084,La Xerea,Ciutat Vella,Entire home/apt,6,2,3,7,"Paid parking on premises, Shampoo, Oven, Host ...",6071.0,2,60,True,0,0,3,3,38,2018-09-03,NaT,93.0,10.0,9.0,10.0,10.0,10.0,9.0,0,0.435115,Spain,Valencia,2020-09-28,True,30


In [115]:
pd.set_option('display.max_columns', None)
df[df['beds'] > 25]

,apartment_id,name,description,host_id,neighbourhood_name,neighbourhood_district,room_type,accommodates,bathrooms,bedrooms,beds,amenities_list,price,minimum_nights,maximum_nights,has_availability,availability_30,availability_60,availability_90,availability_365,number_of_reviews,first_review_date,last_review_date,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,is_instant_bookable,reviews_per_month,country,city,insert_date,is_valid,ocupacion_mes


In [116]:
# df.to_csv('tourist_accommodation_30102025_clean.csv', index=False)  

In [117]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7534 entries, 0 to 7999
Data columns (total 37 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   apartment_id                 7534 non-null   int64         
 1   name                         7531 non-null   object        
 2   description                  7485 non-null   object        
 3   host_id                      7534 non-null   int64         
 4   neighbourhood_name           7534 non-null   object        
 5   neighbourhood_district       4607 non-null   object        
 6   room_type                    7534 non-null   object        
 7   accommodates                 7534 non-null   int64         
 8   bathrooms                    7534 non-null   object        
 9   bedrooms                     7534 non-null   object        
 10  beds                         7534 non-null   object        
 11  amenities_list               7534 non-null   obj

In [118]:
df.isnull().sum()

apartment_id                      0
name                              3
description                      49
host_id                           0
neighbourhood_name                0
neighbourhood_district         2927
room_type                         0
accommodates                      0
bathrooms                         0
bedrooms                          0
beds                              0
amenities_list                    0
price                             0
minimum_nights                    0
maximum_nights                    0
has_availability                  0
availability_30                   0
availability_60                   0
availability_90                   0
availability_365                  0
number_of_reviews                 0
first_review_date              5001
last_review_date               5028
review_scores_rating             83
review_scores_accuracy           92
review_scores_cleanliness        86
review_scores_checkin            97
review_scores_communication 